# Financial — Monte Carlo Portfolio Simulation
Monte Carlo simulation of portfolio returns, Value at Risk, and Sharpe ratio optimization.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import norm
from scipy.optimize import minimize

plt.style.use('dark_background')
plt.rcParams.update({'axes.facecolor': '#0A0A0C', 'figure.facecolor': '#070708',
                     'text.color': '#F0EDE6', 'axes.labelcolor': '#C8A97E',
                     'xtick.color': '#4A4A56', 'ytick.color': '#4A4A56',
                     'axes.edgecolor': '#252528', 'grid.color': '#1A1A1E'})

rng = np.random.default_rng(2024)

# Portfolio assets (representative, not financial advice)
ASSETS = ['Tech ETF', 'S&P 500', 'Bonds', 'Gold', 'Crypto']

# Annual expected returns and volatilities
mu    = np.array([0.18, 0.12, 0.04, 0.07, 0.45])
sigma = np.array([0.22, 0.15, 0.05, 0.14, 0.80])

# Correlation matrix
corr = np.array([
    [1.00,  0.82,  0.10,  0.05,  0.35],
    [0.82,  1.00,  0.15,  0.10,  0.25],
    [0.10,  0.15,  1.00, -0.20, -0.05],
    [0.05,  0.10, -0.20,  1.00,  0.10],
    [0.35,  0.25, -0.05,  0.10,  1.00],
])

# Covariance matrix
cov = np.outer(sigma, sigma) * corr
print('Portfolio config ready ✓')
print(f'Assets: {ASSETS}')

In [ ]:
# ── Monte Carlo portfolio paths ───────────────────────────────────────────────
N_SIM    = 5_000
N_DAYS   = 252    # 1 trading year
INITIAL  = 100_000

# Equal-weight vs optimized allocation
w_equal = np.array([0.20, 0.20, 0.20, 0.20, 0.20])

# Sharpe-optimized weights (maximize Sharpe ratio)
rf = 0.05  # risk-free rate
def neg_sharpe(w):
    port_return = np.dot(w, mu)
    port_vol    = np.sqrt(w @ cov @ w)
    return -(port_return - rf) / port_vol

constraints = ({'type': 'eq', 'fun': lambda w: w.sum() - 1})
bounds = [(0.05, 0.60)] * len(ASSETS)
opt = minimize(neg_sharpe, w_equal, method='SLSQP', bounds=bounds, constraints=constraints)
w_opt = opt.x

def simulate_paths(weights, n_sim=N_SIM, n_days=N_DAYS):
    """GBM simulation with correlated assets."""
    daily_mu  = weights @ mu / 252
    L = np.linalg.cholesky(cov / 252)
    
    paths = np.zeros((n_sim, n_days + 1))
    paths[:, 0] = INITIAL
    
    for t in range(1, n_days + 1):
        z     = rng.standard_normal((len(ASSETS), n_sim))
        shocks = (L @ z).T  # correlated shocks
        port_return = daily_mu + shocks @ weights
        paths[:, t] = paths[:, t-1] * np.exp(port_return)
    
    return paths

paths_equal = simulate_paths(w_equal)
paths_opt   = simulate_paths(w_opt)

print(f'Optimized weights: {dict(zip(ASSETS, w_opt.round(3)))}')
print(f'Expected Sharpe (optimized): {-neg_sharpe(w_opt):.3f}')

In [ ]:
# ── Plot everything ───────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 10))
fig.suptitle('Monte Carlo Portfolio Simulation — 5,000 Paths × 252 Days', color='#C8A97E', fontsize=13, fontweight='bold')
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# Path fans
for i, (paths, weights, label, color) in enumerate([
    (paths_equal, w_equal, 'Equal Weight', '#5AC8FA'),
    (paths_opt,   w_opt,   'Sharpe Optimal', '#C8A97E'),
]):
    ax = fig.add_subplot(gs[0, i])
    # Plot fan (sample 200 paths)
    sample = paths[rng.choice(N_SIM, 200, replace=False)]
    for path in sample:
        ax.plot(path, color=color, alpha=0.04, linewidth=0.8)
    # Percentile bands
    p5, p25, p50, p75, p95 = np.percentile(paths, [5,25,50,75,95], axis=0)
    ax.fill_between(range(N_DAYS+1), p5, p95, alpha=0.15, color=color)
    ax.fill_between(range(N_DAYS+1), p25, p75, alpha=0.25, color=color)
    ax.plot(p50, color=color, linewidth=2.5, label='Median')
    ax.axhline(INITIAL, color='#3A3A42', linestyle='--', linewidth=1)
    ax.set_title(label, color='#F0EDE6')
    ax.set_xlabel('Trading Days'); ax.set_ylabel('Portfolio Value ($)')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x/1e3:.0f}k'))
    ax.grid(True, alpha=0.15)

# Efficient frontier
ax3 = fig.add_subplot(gs[0, 2])
n_port = 2000
rand_w = rng.dirichlet(np.ones(len(ASSETS)), n_port)
port_ret  = rand_w @ mu
port_vol  = np.array([np.sqrt(w @ cov @ w) for w in rand_w])
port_sh   = (port_ret - rf) / port_vol
sc = ax3.scatter(port_vol, port_ret, c=port_sh, cmap='YlOrBr', s=8, alpha=0.6, edgecolors='none')
ax3.scatter([np.sqrt(w_opt @ cov @ w_opt)], [w_opt @ mu], s=120, color='#C8A97E',
            zorder=5, marker='*', label='Optimal')
ax3.scatter([np.sqrt(w_equal @ cov @ w_equal)], [w_equal @ mu], s=80, color='#5AC8FA',
            zorder=5, label='Equal weight')
ax3.set_title('Efficient Frontier', color='#F0EDE6')
ax3.set_xlabel('Annual Volatility'); ax3.set_ylabel('Annual Return')
ax3.legend(fontsize=8, framealpha=0.3); ax3.grid(True, alpha=0.15)
plt.colorbar(sc, ax=ax3, label='Sharpe Ratio')

# Final value distributions
ax4 = fig.add_subplot(gs[1, :2])
final_eq  = paths_equal[:, -1]
final_opt = paths_opt[:, -1]
bins = np.linspace(min(final_eq.min(), final_opt.min()), 
                   min(final_eq.max(), final_opt.max(), 500_000), 60)
ax4.hist(final_eq,  bins=bins, alpha=0.5, color='#5AC8FA', label='Equal weight',  density=True)
ax4.hist(final_opt, bins=bins, alpha=0.5, color='#C8A97E', label='Sharpe optimal', density=True)
# VaR lines
for finals, color, label in [(final_eq,'#5AC8FA','EW'),(final_opt,'#C8A97E','Opt')]:
    var95 = np.percentile(finals, 5)
    ax4.axvline(var95, color=color, linestyle=':', linewidth=1.5, label=f'VaR 95% ({label}): ${var95:,.0f}')
ax4.axvline(INITIAL, color='#3A3A42', linestyle='--', linewidth=1, label='Initial $100k')
ax4.set_title('Final Portfolio Value Distribution', color='#F0EDE6')
ax4.set_xlabel('Value ($)'); ax4.legend(fontsize=7, framealpha=0.3); ax4.grid(True, alpha=0.15)
ax4.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x/1e3:.0f}k'))

# Allocation comparison
ax5 = fig.add_subplot(gs[1, 2])
x   = np.arange(len(ASSETS))
ax5.bar(x-0.2, w_equal, 0.35, color='#5AC8FA', alpha=0.8, label='Equal')
ax5.bar(x+0.2, w_opt,   0.35, color='#C8A97E', alpha=0.9, label='Optimized')
ax5.set_xticks(x); ax5.set_xticklabels(ASSETS, rotation=20, ha='right', fontsize=8)
ax5.set_title('Allocation Comparison', color='#F0EDE6')
ax5.set_ylabel('Weight'); ax5.legend(fontsize=8, framealpha=0.3); ax5.grid(True, alpha=0.15, axis='y')

plt.savefig('monte_carlo.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary
print('\n── Results ────────────────────────────────────────')
for label, finals in [('Equal weight', final_eq), ('Sharpe optimal', final_opt)]:
    print(f'{label}:')
    print(f'  Median return : {(np.median(finals)/INITIAL-1)*100:.1f}%')
    print(f'  VaR 95%       : ${np.percentile(finals,5):,.0f}')
    print(f'  Win rate      : {(finals>INITIAL).mean()*100:.1f}% of paths profitable')
    print(f'  Best/Worst    : ${finals.max():,.0f} / ${finals.min():,.0f}')